In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import os

os.chdir('/Users/syedamishrasaiara/pythonprojects/SQL_Analytics')
DATA = "data/yellow_tripdata_2024-01.parquet"
os.makedirs('charts', exist_ok=True)
print("Ready!") 

Ready!


## Driver & Vendor Performance — NYC Yellow Taxi (January 2024)
Analyzing performance differences between vendors and identifying 
trip characteristics that predict higher earnings.

In [2]:
# Vendor comparison
vendors = duckdb.sql(f"""
    SELECT
        CASE VendorID WHEN 1 THEN 'Creative Mobile Tech' ELSE 'VeriFone' END AS vendor,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_amount), 2) AS avg_tip,
        ROUND(AVG(passenger_count), 1) AS avg_passengers
    FROM '{DATA}'
    WHERE fare_amount > 0 AND trip_distance > 0
    GROUP BY VendorID
""").df()

vendors

,vendor,total_trips,avg_distance,avg_fare,avg_tip,avg_passengers
0,VeriFone,2173813,3.93,18.83,3.49,1.4
1,VeriFone,260,11.35,46.90,0.00,NaN
2,Creative Mobile Tech,695641,3.12,17.47,3.12,1.2


In [3]:
# Earnings per mile (efficiency metric)
efficiency = duckdb.sql(f"""
    SELECT
        CASE VendorID WHEN 1 THEN 'Creative Mobile Tech' ELSE 'VeriFone' END AS vendor,
        ROUND(AVG(total_amount / trip_distance), 2) AS revenue_per_mile,
        ROUND(AVG(tip_amount / NULLIF(fare_amount, 0)) * 100, 1) AS avg_tip_pct
    FROM '{DATA}'
    WHERE fare_amount > 0 AND trip_distance > 0.5
    GROUP BY VendorID
""").df()

efficiency

,vendor,revenue_per_mile,avg_tip_pct
0,VeriFone,12.05,20.6
1,VeriFone,5.92,0.0
2,Creative Mobile Tech,11.93,23.0


In [4]:
# Top earning trip profiles (CTE + ranking)
profiles = duckdb.sql(f"""
    WITH trip_metrics AS (
        SELECT
            trip_distance,
            fare_amount,
            tip_amount,
            total_amount,
            EXTRACT(HOUR FROM tpep_pickup_datetime) AS hour,
            DAYNAME(tpep_pickup_datetime) AS day_name,
            ROUND(tip_amount / NULLIF(fare_amount,0) * 100, 1) AS tip_pct,
            NTILE(4) OVER (ORDER BY total_amount DESC) AS earnings_quartile
        FROM '{DATA}'
        WHERE fare_amount > 0 AND trip_distance > 0
    )
    SELECT
        earnings_quartile,
        COUNT(*) AS trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_pct), 1) AS avg_tip_pct,
        ROUND(AVG(total_amount), 2) AS avg_total
    FROM trip_metrics
    GROUP BY earnings_quartile
    ORDER BY earnings_quartile
""").df()

profiles

,earnings_quartile,trips,avg_distance,avg_fare,avg_tip_pct,avg_total
0,1,717429,9.23,40.25,21.2,54.99
1,2,717429,3.17,15.76,21.4,23.71
2,3,717428,1.69,10.74,22.8,17.74
3,4,717428,0.84,7.25,20.2,12.95


## Key Findings

- **VeriFone dominates by volume**: VeriFone processed 2.17M trips vs 
  Creative Mobile Tech's 695K — roughly a 3:1 ratio
- **Creative Mobile tips better**: Creative Mobile Tech drivers earn a 
  23.0% avg tip vs VeriFone's 20.6%, despite charging lower avg fares 
  ($17.47 vs $18.83)
- **Revenue per mile is nearly identical**: VeriFone earns $12.05/mile, 
  Creative Mobile earns $11.93/mile — suggesting similar operational 
  efficiency despite different trip volumes
- **Anomaly detected**: A small group of 260 VeriFone trips show an avg 
  fare of $46.90 with $0 tip and no passenger count — likely corporate 
  or dispatch bookings logged differently in the system
- **Top earning trips travel 11× further**: Q1 (highest earnings) trips 
  average 9.23 miles vs Q4's 0.84 miles — distance is the single biggest 
  driver of total fare
- **Tip % is consistent across quartiles**: All quartiles tip between 
  20–23%, meaning tipping behavior doesn't change much regardless of 
  fare size — high earners win on distance, not tips
- **Best trip profile**: Long-distance trips (9+ miles) averaging $54.99 
  total are the highest-value segment — airport runs and outer borough 
  trips likely make up most of Q1